# PV181 RNG — Python solution

Solution notebook for the current `PV181_RNG_python.ipynb`.

## PRNG determinism (Tasks 1–6)

In [ ]:
import random

rnd_bytes = random.randbytes(10)
print(rnd_bytes)
print(rnd_bytes.hex())

random.seed(1)
first = random.randbytes(10)
random.seed(1)
second = random.randbytes(10)
assert first == second
print('Fixed-seed output:', first.hex())

random.seed(1)
state = random.getstate()
from_seed = random.randbytes(10)
random.setstate(state)
from_state = random.randbytes(10)
assert from_seed == from_state
print('State restored:', from_state.hex())

## Small-seed attack (Task 7)

In [ ]:
known_half = bytes.fromhex('73a9bef499bbf4dc')

for seed in range(10):
    random.seed(seed)
    candidate = random.randbytes(16)
    if candidate[:len(known_half)] == known_half:
        print(f'Seed: {seed}')
        print(f'Full key: {candidate.hex()}')

## Time-based seed attack (Task 8)

In [ ]:
import datetime
import time

t_generate = int(time.time())
random.seed(t_generate)
rnd_bytes = random.randbytes(10)
print('Bytes generated:', rnd_bytes.hex())

# In a realistic attack, the attacker knows an approximate time window.
window = range(t_generate - 60, t_generate + 61)
for candidate_seed in window:
    random.seed(candidate_seed)
    if random.randbytes(10) == rnd_bytes:
        print('Recovered seed:', candidate_seed)
        print('UTC time:', datetime.datetime.fromtimestamp(candidate_seed, datetime.timezone.utc).isoformat())

## LCG (Tasks 9–13)

In [ ]:
class LCG:
    def __init__(self, a, c, m):
        self.a = a
        self.c = c
        self.m = m
        self.srand(0)

    def srand(self, seed):
        self.state = seed

    def rand(self):
        self.state = (self.state * self.a + self.c) % self.m
        return self.state

a, c, m = 1103515245, 12345, 2**31
ansi_rand = LCG(a, c, m)
ansi_rand.srand(0)
rnd_values = [ansi_rand.rand() for _ in range(10)]
print(rnd_values)
assert rnd_values[:2] == [12345, 1406932606]

# The seed whose first generated value is 1406932606 is 12345.
ansi_rand.srand(12345)
assert ansi_rand.rand() == 1406932606
print('Seed for first value 1406932606:', 12345)

In [ ]:
# Reverse the ANSI LCG. This requires gcd(a, m) == 1.
a_back = pow(a, -1, m)
c_back = (-c * a_back) % m

backward = LCG(a_back, c_back, m)
backward.srand(1406932606)
reverse_outputs = [backward.rand() for _ in range(3)]
print('Reverse outputs:', reverse_outputs)
assert reverse_outputs == [12345, 0, 2088216195]
print('Missing states in forward order:', [2088216195, 0])

In [ ]:
# Java Random uses a scrambled 48-bit seed and returns the high 32 bits.
JAVA_MULTIPLIER = 25214903917
JAVA_INCREMENT = 11
JAVA_MASK = (1 << 48) - 1

def java_random_ints(seed, count):
    state = (seed ^ JAVA_MULTIPLIER) & JAVA_MASK
    values = []
    for _ in range(count):
        state = (state * JAVA_MULTIPLIER + JAVA_INCREMENT) & JAVA_MASK
        values.append(state >> 16)
    return values

java_values = java_random_ints(1, 10)
print(java_values)
assert java_values[0] == 3139097971

## Small-state attack (Tasks 14–17)

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import os

def SHA1(message):
    digest = hashes.Hash(hashes.SHA1())
    digest.update(message)
    return digest.finalize()

def SHA256(message):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(message)
    return digest.finalize()

class SmallStatePRNG:
    def __init__(self, state_size=1):
        self.state_size = state_size
        self.srand(os.urandom(16))

    def srand(self, seed):
        self.state = SHA256(seed)[:self.state_size]

    def rand_bytes(self, num_bytes=10):
        result = SHA256(self.state)[:num_bytes]
        self.state = SHA1(self.state)[:self.state_size]
        return result

# Explore the state transition until a state repeats.
rng = SmallStatePRNG()
seen = {}
state_sequence = []
while rng.state not in seen:
    seen[rng.state] = len(state_sequence)
    state_sequence.append(rng.state)
    rng.rand_bytes(5)
cycle_start = seen[rng.state]
print('Preperiod:', cycle_start)
print('Cycle length:', len(state_sequence) - cycle_start)

# Enumerate every possible first 16-byte block.
all_keys = []
for value in range(256):
    rng.state = bytes([value])
    all_keys.append(rng.rand_bytes(16))
print('Candidate keys:', len(all_keys))
assert len(all_keys) == 256

In [ ]:
def encrypt_ECB(key, message):
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    encryptor = cipher.encryptor()
    return encryptor.update(message) + encryptor.finalize()

def decrypt_ECB(key, ciphertext):
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    decryptor = cipher.decryptor()
    return decryptor.update(ciphertext) + decryptor.finalize()

rng = SmallStatePRNG()
ct1 = encrypt_ECB(rng.rand_bytes(16), b'arbitrarymessage')

for key in all_keys:
    if decrypt_ECB(key, ct1) == b'arbitrarymessage':
        print('Recovered K1:', key.hex())

ct2 = bytes.fromhex('b0c51f35872c4a4832d15c38b0d42d59')
for key in all_keys:
    plaintext = decrypt_ECB(key, ct2)
    if all(32 <= byte <= 126 for byte in plaintext):
        print('Recovered PT2:', plaintext)

## Operating-system randomness and bit histograms (Tasks 18–20)

In [ ]:
import secrets

print(os.urandom(10).hex())
print(secrets.token_bytes(10).hex())

# Prefer os.urandom/secrets to reading /dev/random directly in a notebook.

In [ ]:
def histogram(rnd_bytes, i, j):
    if i == j or not (0 <= i < 8 and 0 <= j < 8):
        raise ValueError('i and j must be distinct bit positions in 0..7')

    mask = (1 << i) | (1 << j)
    hist = {0: 0, 1 << i: 0, 1 << j: 0, mask: 0}
    for byte in rnd_bytes:
        hist[byte & mask] += 1
    return hist

test_bytes = bytes([0, 1, 2, 3, 0, 1, 2, 3])
result = histogram(test_bytes, 0, 1)
print(result)
assert result == {0: 2, 1: 2, 2: 2, 3: 2}

In [ ]:
trng_bytes = os.urandom(1000)
for i, j in [(0, 1), (2, 5), (6, 7)]:
    counts = histogram(trng_bytes, i, j)
    print((i, j), counts)

ansi_rand.srand(0)
lcg_bytes = bytes(ansi_rand.rand() % 256 for _ in range(1000))
for i, j in [(0, 1), (2, 5), (6, 7)]:
    counts = histogram(lcg_bytes, i, j)
    print('LCG', (i, j), counts)